In [1]:
import pandas as pd 
import requests
from jsonschema import validate
import json
import fastparquet

In [2]:
pd.set_option('display.width', None)

In [3]:
countriesDf = pd.DataFrame(pd.read_parquet('../dataRawBeforeBlob/countries/euCountriesRaw.parquet'))
countriesDf

,Code,Title,ParentDimension,Dimension,ParentCode,ParentTitle
0,AUT,Austria,REGION,COUNTRY,EUR,Europe
1,BEL,Belgium,REGION,COUNTRY,EUR,Europe
2,BGR,Bulgaria,REGION,COUNTRY,EUR,Europe
3,CYP,Cyprus,REGION,COUNTRY,EUR,Europe
4,CZE,Czechia,REGION,COUNTRY,EUR,Europe
5,DEU,Germany,REGION,COUNTRY,EUR,Europe
6,DNK,Denmark,REGION,COUNTRY,EUR,Europe
7,ESP,Spain,REGION,COUNTRY,EUR,Europe
8,EST,Estonia,REGION,COUNTRY,EUR,Europe
9,FIN,Finland,REGION,COUNTRY,EUR,Europe


In [4]:
alcoholIndicatorsDf = pd.DataFrame(pd.read_parquet('../dataRawBeforeBlob/indicators/alcoholIndicators.parquet',engine='fastparquet'))
alcoholIndicatorsDf

,IndicatorCode,IndicatorName,Language,Category
0,SA_0000001550,Excise tax on alcoholic beverages,EN,Alcohol Control Policies
1,SA_0000001507,Advertising restrictions on national television,EN,Alcohol Control Policies
2,SA_0000001699,Age limits off-premise sales,EN,Alcohol Control Policies
3,SA_0000001751,"Alcohol, average daily intake in grams among d...",EN,Alcohol levels of consumption
4,SA_0000001833,"Alcohol-attributable DALYs per 100,000 people ...",EN,Harms and consequences of alcohol
5,SA_0000001844,Alcohol-attributable all-cause deaths (all age...,EN,Harms and consequences of alcohol


In [5]:
baseURL= 'https://ghoapi.azureedge.net/api/'

In [33]:
def getData(baseurl,indicatorCode:str):
    indicatorCode = indicatorCode.strip()
    response = requests.get(baseurl+f"{indicatorCode}")
    if response.status_code == 200 and response.headers.get('Content-Type').startswith('application/json') :
        return response.json()
    else:
        return 'Something went wrong with request'
    

In [14]:
def saveToParquet(df: pd.DataFrame,filename:str):
    df.to_parquet(f'../dataRawBeforeBlob/alcoholData/{filename}.parquet',engine='fastparquet')

In [9]:
firstIndicator = alcoholIndicatorsDf.iloc[0]
firstIndicator

IndicatorCode                        SA_0000001550
IndicatorName    Excise tax on alcoholic beverages
Language                                        EN
Category                  Alcohol Control Policies
Name: 0, dtype: object

In [11]:
firstIndicatorData = getData(baseURL,firstIndicator['IndicatorCode'])
firstIndicatorData = firstIndicatorData['value']

In [15]:
firstIndicatorDataDataFrame = pd.DataFrame(firstIndicatorData)
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame.loc[firstIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
firstIndicatorDataDataFrame = firstIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
firstIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
2,3291727,SA_0000001550,POL,2016,ALCOHOLTYPE_SA_SPIRITS,Yes
8,356170,SA_0000001550,BEL,2019,ALCOHOLTYPE_SA_SPIRITS,Yes
20,9462260,SA_0000001550,LVA,2016,ALCOHOLTYPE_SA_WINE,Yes
30,4380483,SA_0000001550,AUT,2016,ALCOHOLTYPE_SA_BEER,Yes
48,2431283,SA_0000001550,SVK,2016,ALCOHOLTYPE_SA_WINE,Yes
...,...,...,...,...,...,...
1005,6017451,SA_0000001550,FRA,2016,ALCOHOLTYPE_SA_SPIRITS,Yes
1013,6669386,SA_0000001550,DEU,2016,ALCOHOLTYPE_SA_SPIRITS,Yes
1025,8198213,SA_0000001550,POL,2019,ALCOHOLTYPE_SA_BEER,Yes
1026,8242761,SA_0000001550,ROU,2019,ALCOHOLTYPE_SA_WINE,Yes


In [16]:
saveToParquet(firstIndicatorDataDataFrame,'exciseTaxOnBeverages')

In [17]:
secondIndicator = alcoholIndicatorsDf.iloc[1]
secondIndicator

IndicatorCode                                      SA_0000001507
IndicatorName    Advertising restrictions on national television
Language                                                      EN
Category                                Alcohol Control Policies
Name: 1, dtype: object

In [19]:
secondIndicatorData = getData(baseURL,secondIndicator['IndicatorCode'])
secondIndicatorData = secondIndicatorData['value']

In [22]:
secondIndicatorDataDataFrame = pd.DataFrame(secondIndicatorData)
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame.loc[secondIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
secondIndicatorDataDataFrame = secondIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
secondIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
21,9077496,SA_0000001507,MLT,2016,ADVERTISINGTYPE_SA_WINE_ADS,partial restriction time/content
23,7617895,SA_0000001507,CYP,2016,ADVERTISINGTYPE_SA_BEER_ADS,partial restriction time/place/content
26,6397335,SA_0000001507,SVK,2019,ADVERTISINGTYPE_SA_SPIRITS_ADS,No data
33,2602433,SA_0000001507,SVK,2016,ADVERTISINGTYPE_SA_BEER_ADS,partial restriction place
55,938484,SA_0000001507,SVN,2016,ADVERTISINGTYPE_SA_WINE_ADS,partial restriction time/place/content
...,...,...,...,...,...,...
1025,9775184,SA_0000001507,HRV,2016,ADVERTISINGTYPE_SA_BEER_ADS,no restriction
1030,8697697,SA_0000001507,CYP,2016,ADVERTISINGTYPE_SA_SPIRITS_ADS,partial restriction time/place/content
1037,1191037,SA_0000001507,ROU,2019,ADVERTISINGTYPE_SA_SPIRITS_ADS,"Partial statutory restriction - time, place, c..."
1039,5379842,SA_0000001507,LVA,2016,ADVERTISINGTYPE_SA_WINE_ADS,partial restriction time/content


In [23]:
saveToParquet(secondIndicatorDataDataFrame,'adverstingAlcohol')

In [24]:
thirdIndicator = alcoholIndicatorsDf.iloc[2]
thirdIndicator

IndicatorCode                   SA_0000001699
IndicatorName    Age limits off-premise sales
Language                                   EN
Category             Alcohol Control Policies
Name: 2, dtype: object

In [27]:
thirdIndicatorData = getData(baseURL,thirdIndicator['IndicatorCode'])
thirdIndicatorData = thirdIndicatorData['value']

In [28]:
thirdIndicatorDataDataFrame = pd.DataFrame(thirdIndicatorData)
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame.loc[thirdIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
thirdIndicatorDataDataFrame = thirdIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
thirdIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
5,5278753,SA_0000001699,HRV,2016,ALCOHOLTYPE_SA_BEER,18
6,457389,SA_0000001699,HUN,2019,ALCOHOLTYPE_SA_WINE,18
10,3215887,SA_0000001699,CZE,2016,ALCOHOLTYPE_SA_WINE,18
35,64989,SA_0000001699,BEL,2016,ALCOHOLTYPE_SA_WINE,16
47,4544787,SA_0000001699,GRC,2019,ALCOHOLTYPE_SA_BEER,18
...,...,...,...,...,...,...
1008,2829851,SA_0000001699,LUX,2019,ALCOHOLTYPE_SA_WINE,16
1018,6168977,SA_0000001699,GRC,2016,ALCOHOLTYPE_SA_SPIRITS,18
1029,7913136,SA_0000001699,FRA,2016,ALCOHOLTYPE_SA_BEER,18
1030,5316533,SA_0000001699,DEU,2019,ALCOHOLTYPE_SA_SPIRITS,18


In [29]:
saveToParquet(thirdIndicatorDataDataFrame,'alcoholOffPremiseAgeSale')

In [30]:
fourthIndicator = alcoholIndicatorsDf.iloc[3]
fourthIndicator

IndicatorCode                                        SA_0000001751
IndicatorName    Alcohol, average daily intake in grams among d...
Language                                                        EN
Category                             Alcohol levels of consumption
Name: 3, dtype: object

In [34]:
fourthIndicatorData = getData(baseURL,fourthIndicator['IndicatorCode'])
fourthIndicatorData = fourthIndicatorData['value']

In [49]:
fourthIndicatorDataDataFrame = pd.DataFrame(fourthIndicatorData)
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame.loc[fourthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame.loc[fourthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].astype('string')
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
fourthIndicatorDataDataFrame['Value'] = fourthIndicatorDataDataFrame['Value'].astype('float')
fourthIndicatorDataDataFrame = fourthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
fourthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
0,947,SA_0000001751,IRL,2006,SEX_BTSX,37.0
5,4696,SA_0000001751,DNK,2003,SEX_BTSX,31.8
27,19873,SA_0000001751,HRV,2000,SEX_BTSX,33.9
57,43560,SA_0000001751,GRC,2014,SEX_BTSX,23.2
90,70917,SA_0000001751,NLD,2017,SEX_BTSX,26.2
...,...,...,...,...,...,...
11675,10117228,SA_0000001751,NLD,2004,SEX_BTSX,29.8
11685,10122469,SA_0000001751,FRA,2010,SEX_BTSX,34.3
11691,10127696,SA_0000001751,AUT,2016,SEX_BTSX,32.5
11741,10167236,SA_0000001751,SVK,2019,SEX_BTSX,31.3


In [50]:
saveToParquet(fourthIndicatorDataDataFrame,'alcoholAverageDailyIntakeBothSex')

In [51]:
fifthIndicator = alcoholIndicatorsDf.iloc[4]
fifthIndicator

IndicatorCode                                        SA_0000001833
IndicatorName    Alcohol-attributable DALYs per 100,000 people ...
Language                                                        EN
Category                         Harms and consequences of alcohol
Name: 4, dtype: object

In [53]:
fifthIndicatorData = getData(baseURL,fifthIndicator['IndicatorCode'])
fifthIndicatorData = fifthIndicatorData['value']

In [59]:
fifthIndicatorDataDataFrame = pd.DataFrame(fifthIndicatorData)
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame.loc[fifthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame[(fifthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX') &( fifthIndicatorDataDataFrame['Dim2'] == 'AGEGROUP_YEARSALL')]
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].astype('string')
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].str.split(' ').str[0]
fifthIndicatorDataDataFrame['Value'] = fifthIndicatorDataDataFrame['Value'].astype('float')
fifthIndicatorDataDataFrame = fifthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]
fifthIndicatorDataDataFrame

,Id,IndicatorCode,SpatialDim,TimeDim,Dim1,Value
171,5639380,SA_0000001833,HUN,2019,SEX_BTSX,2172.1
178,3007858,SA_0000001833,CZE,2019,SEX_BTSX,2096.5
187,9865019,SA_0000001833,DEU,2019,SEX_BTSX,1395.3
203,9027965,SA_0000001833,AUT,2019,SEX_BTSX,1470.1
215,4571229,SA_0000001833,ROU,2019,SEX_BTSX,3033.3
227,894857,SA_0000001833,SVN,2019,SEX_BTSX,2047.7
238,9966510,SA_0000001833,POL,2019,SEX_BTSX,2628.6
242,4192046,SA_0000001833,FIN,2019,SEX_BTSX,1660.6
295,5463009,SA_0000001833,BGR,2019,SEX_BTSX,1909.2
314,5084381,SA_0000001833,DNK,2019,SEX_BTSX,1330.4


In [60]:
saveToParquet(fifthIndicatorDataDataFrame,'DALYSBothSexesAllAges')

In [61]:
sixthIndicator = alcoholIndicatorsDf.iloc[5]
sixthIndicator

IndicatorCode                                        SA_0000001844
IndicatorName    Alcohol-attributable all-cause deaths (all age...
Language                                                        EN
Category                         Harms and consequences of alcohol
Name: 5, dtype: object

In [65]:
sixthIndicatorData = getData(baseURL,sixthIndicator['IndicatorCode'])
sixthIndicatorData = sixthIndicatorData['value']

In [69]:
sixthIndicatorDataDataFrame = pd.DataFrame(sixthIndicatorData)
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame.loc[sixthIndicatorDataDataFrame['SpatialDim'].isin(countriesDf['Code'])]
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame.loc[sixthIndicatorDataDataFrame['Dim1'] == 'SEX_BTSX']
sixthIndicatorDataDataFrame = sixthIndicatorDataDataFrame[['Id','IndicatorCode','SpatialDim','TimeDim','Dim1','Value']]

In [ ]:
saveToParquet(sixthIndicatorDataDataFrame,'allAlcoholCauseDeathsBothSexesAllAGes')